# Ephemeris Mean Parameters
Notebook to aggregate per-observation ephemeris statistics.
This reuses the directory + ephemeris helpers originally developed in Exploring.ipynb.

In [2]:
from pathlib import Path
from datetime import datetime, timedelta
import io
import os
import re

import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.time import Time

In [2]:
# Directory + ephemeris helpers brought from Exploring.ipynb
def search_directories(base_path):
    directories = []
    for dir in os.listdir(base_path):
        if 'DS_Store' in dir:
            continue
        directories.append(base_path + '/' + dir)
    directories = sorted(directories, key=lambda x: x.split('/')[1])
    return directories

def walk_directories(base_path):
    directories = []
    for root, dirs, files in os.walk(base_path):
        for dir in dirs:
            directories.append(os.path.join(root, dir))
    return directories

def files_fiberA_in_visit(directory):
    files = []
    directs = walk_directories(directory)
    for dir in directs:
        for file in os.listdir(dir):
            if 'FINAL_A' in file:
                files.append(os.path.join(dir, file))
    return files

def date_of_obs(directory):
    return directory.split('/')[1]


In [3]:
def nearest_ephemeris_row(ephemeris, middle, mjd_col=None):
    if mjd_col is None:
        for cand in ("MJD", "mjd", "ephemeris('MJD')", "Mjd"):
            if cand in ephemeris.columns:
                mjd_col = cand
                break
        if mjd_col is None:
            raise KeyError("Could not find an MJD column. Pass mjd_col='...'.")
    mjd_vals = pd.to_numeric(ephemeris[mjd_col], errors='coerce').to_numpy()
    if np.isnan(mjd_vals).all():
        raise ValueError(f"Column {mjd_col!r} contains no numeric values.")
    idx_pos = int(np.nanargmin(np.abs(mjd_vals - float(middle))))
    return ephemeris.iloc[idx_pos]

def read_fits(file, ephemeris):
    with fits.open(file) as hdul:
        start = hdul[0].header['MJD-OBS']
        end = hdul[0].header['MJD-END']
        middle = (end + start) / 2
        row = nearest_ephemeris_row(ephemeris, middle)
    return row

def _normalize_yyyymmdd(date_obs):
    m = re.search(r'(\d{4})[-/]?(\d{2})[-/]?(\d{2})', date_obs)
    if not m:
        raise ValueError(f"Couldn't find a YYYYMMDD date in {date_obs!r}")
    y, mo, d = m.groups()
    datetime.strptime(f"{y}{mo}{d}", '%Y%m%d')
    return f'{y}{mo}{d}'

def find_ephemeris_path(date_obs, base_dir='Ephemeris', filename_pattern='horizons_results_{date}.txt', try_prev_day=True):
    ymd = _normalize_yyyymmdd(date_obs)
    d0 = datetime.strptime(ymd, '%Y%m%d').date()
    candidates = [d0]
    if try_prev_day:
        candidates.append(d0 - timedelta(days=1))
    tried = []
    for d in candidates:
        p = Path(base_dir) / filename_pattern.format(date=d.strftime('%Y%m%d'))
        tried.append(str(p))
        if p.exists():
            return p
    raise FileNotFoundError("Ephemeris file not found. Tried:\n  " + "\n  ".join(tried))

def utc_to_mjd(iso_utc):
    return Time(iso_utc, scale='utc').mjd

def horizons_to_dataframe(path):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        txt = f.read()
    try:
        block = txt.split('$$SOE', 1)[1].split('$$EOE', 1)[0]
    except IndexError:
        raise ValueError('Could not find $$SOE/$$EOE block.')
    lines = [ln for ln in block.splitlines() if ln.strip() and not ln.lstrip().startswith('>')]
    block = '\n'.join(lines)
    df = pd.read_csv(io.StringIO(block), header=None, engine='python', skipinitialspace=True, dtype=str)
    if df.shape[1] >= 15:
        last = df.columns[-1]
        if df[last].isna().all() or df[last].astype(str).str.strip().eq('').all():
            df = df.iloc[:, :-1]
    df.columns = [
        'datetime_utc','jd_ut','sun_presence','moon_presence','ra_hms','dec_dms',
        'dra_cosdec_arcsec_per_hr','ddec_arcsec_per_hr','t_mag','n_mag',
        'r_au','rdot_km_s','delta_au','deldot_km_s'
    ]
    for c in ['datetime_utc','sun_presence','moon_presence','ra_hms','dec_dms']:
        df[c] = df[c].astype(str).str.strip()
    df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], format='%Y-%b-%d %H:%M:%S')
    for col in ['jd_ut','dra_cosdec_arcsec_per_hr','ddec_arcsec_per_hr','t_mag','n_mag',
                'r_au','rdot_km_s','delta_au','deldot_km_s']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    def ra_to_deg(s):
        h, m, sec = s.split()
        return (float(h) + float(m)/60 + float(sec)/3600) * 15.0
    def dec_to_deg(s):
        d, m, sec = s.split()
        sign = -1 if d.startswith('-') else 1
        d = abs(float(d))
        return sign * (d + float(m)/60 + float(sec)/3600)
    df['ra_deg'] = df['ra_hms'].map(ra_to_deg)
    df['dec_deg'] = df['dec_dms'].map(dec_to_deg)
    df['MJD'] = utc_to_mjd(df['datetime_utc'])
    return df

def fits_in_vistit(directory):
    files = files_fiberA_in_visit(directory)
    obs_info = {}
    ephemeris = horizons_to_dataframe(find_ephemeris_path(directory))
    for file in files:
        if 'sky' in file or 'solar' in file:
            continue
        obs_info[file] = read_fits(file, ephemeris)
    return obs_info


In [4]:
visit_directories = search_directories('data')
summary_rows = []
for visit_dir in visit_directories:
    obs_info = fits_in_vistit(visit_dir)
    #take out the visit with sky in the name:
    # obs_info = [obs_info[info] for info in obs_info.keys() if 'sky' not in info['filename']]
    if not obs_info:
        continue
    info_df = pd.DataFrame(obs_info).T
    info_numeric = info_df.apply(pd.to_numeric, errors='coerce')
    numeric_means = info_numeric.mean(skipna=True)
    row = {
        'date_obs': date_of_obs(visit_dir),
        'n_exposures': len(obs_info),
        'ephemeris_file': str(find_ephemeris_path(visit_dir))
    }
    # include every numeric mean that evaluated to a number
    for col, val in numeric_means.items():
        if pd.notna(val):
            row[f'mean_{col}'] = val
    # ensure key ephemeris quantities are present, even if NaN
    for essential in ['r_au', 'rdot_km_s', 'delta_au', 'deldot_km_s']:
        key = f'mean_{essential}'
        row.setdefault(key, numeric_means.get(essential, np.nan))
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).set_index('date_obs').sort_index()
summary_df

,n_exposures,ephemeris_file,mean_jd_ut,mean_dra_cosdec_arcsec_per_hr,mean_ddec_arcsec_per_hr,mean_t_mag,mean_r_au,mean_rdot_km_s,mean_delta_au,mean_deldot_km_s,mean_ra_deg,mean_dec_deg,mean_MJD
date_obs,,,,,,,,,,,,,
20250902,9,Ephemeris/horizons_results_20250902.txt,2.460922e+06,-86.604644,17.416910,16.104111,2.454532,-51.558241,2.569491,-4.508111,231.187432,-14.240963,60921.032022
20250903,12,Ephemeris/horizons_results_20250903.txt,2.460923e+06,-85.887600,17.481641,16.077917,2.424808,-51.302421,2.566766,-4.284137,230.597541,-14.124474,60922.032697
20250908,3,Ephemeris/horizons_results_20250908.txt,2.460928e+06,-82.382733,17.732870,15.946000,2.278682,-49.853836,2.554724,-3.473835,227.725072,-13.537290,60927.033333
20250909,6,Ephemeris/horizons_results_20250909.txt,2.460929e+06,-81.768533,17.758825,15.920167,2.250393,-49.531124,2.552565,-3.398684,227.173499,-13.420788,60928.018981
20250910,3,Ephemeris/horizons_results_20250910.txt,2.460929e+06,-81.194867,17.778997,15.894000,2.222456,-49.196800,2.550462,-3.349223,226.629809,-13.304780,60928.998843
20250911,6,Ephemeris/horizons_results_20250911.txt,2.460931e+06,-80.475633,17.816095,15.866500,2.193854,-48.837313,2.548336,-3.277404,226.073912,-13.184949,60930.009144
20250912,3,Ephemeris/horizons_results_20250912.txt,2.460932e+06,-79.840800,17.842257,15.840000,2.165861,-48.467476,2.546261,-3.244526,225.530488,-13.066643,60931.005324
20250915,4,Ephemeris/horizons_results_20250915.txt,2.460934e+06,-77.999775,17.911008,15.758500,2.083180,-47.258276,2.540054,-3.270981,223.926281,-12.710649,60933.995486
20250916,4,Ephemeris/horizons_results_20250916.txt,2.460936e+06,-77.307800,17.944640,15.730500,2.055533,-46.810123,2.537914,-3.296187,223.389095,-12.589180,60935.013194


In [5]:
output_path = Path('Ephemeris/ephemeris_means_by_observation.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path)
print(f'Saved {len(summary_df)} rows to {output_path}')

Saved 22 rows to Ephemeris/ephemeris_means_by_observation.csv


# This is to create the seeing and z tables

In [ ]:
def search_directories(base_path):
    directories = []
    for dir in os.listdir(base_path):
        if 'DS_Store' in dir:
            continue
        directories.append(base_path + '/' + dir)
    directories = sorted(directories, key=lambda x: x.split('/')[1])
    return directories

def walk_directories(base_path):
    directories = []
    for root, dirs, files in os.walk(base_path):
        for dir in dirs:
            directories.append(os.path.join(root, dir))
    return directories

def files_fiberA_in_visit(directory):
    files = []
    directs = walk_directories(directory)
    for dir in directs:
        for file in os.listdir(dir):
            if 'sky' in file:
                continue
            if 'FINAL_A' in file:
                files.append(os.path.join(dir, file))
    return files

#Return the fit and the nearest ephemeris row
def read_fits(file):
    with fits.open(file) as hdul:
        if 'U1' in hdul[0].header['TELESCOP']:
            numb = 1
        elif 'U2' in hdul[0].header['TELESCOP']:
            numb = 2
        elif 'U3' in hdul[0].header['TELESCOP']:
            numb = 3
        elif 'U4' in hdul[0].header['TELESCOP']:
            numb = 4
        seeing = np.mean([hdul[0].header[f'ESO TEL{numb} AMBI FWHM END'], hdul[0].header[f'ESO TEL{numb} AMBI FWHM START']])
        Z = 90 - hdul[0].header[f'ESO TEL{numb} ALT']
    return seeing, Z, numb


#open all the exposures in a visit directory
visit_directories = search_directories('../data') 
def fits_in_vistit(directory):
    files = files_fiberA_in_visit(directory)
    seeing_list = []
    Z_list = []
    numbers = []
    for i in files:
        print(i)
        seeing, Z, numb = read_fits(i)
        seeing_list.append(seeing)
        Z_list.append(Z)
        numbers.append(numb)
    print(numbers)
    return min(seeing_list), max(seeing_list), min(Z_list), max(Z_list), numbers

dic = {}
for i in visit_directories:
    name = i.split('/')[2]
    min_seeing, max_seeing, min_Z, max_Z, numbers = fits_in_vistit(i)
    dic[name] = {'min_seeing': min_seeing, 'max_seeing': max_seeing, 'min_Z': min_Z, 'max_Z': max_Z, 'UT': numbers[0]}

#lets save it as a csv file
df = pd.DataFrame.from_dict(dic, orient='index')
df.to_csv('../tables_random/seeing_Z_table.csv')

['../data/20250908', '../data/20250909', '../data/20251127', '../data/20251126', '../data/20250922', '../data/20251204', '../data/20250915', '../data/20250912', '../data/20250924', '../data/20250923', '../data/20250902', '../data/20250903', '../data/20251125', '../data/20250919', '../data/20250921', '../data/20250917', '../data/20250910', '../data/20250911', '../data/20251206', '../data/20250916', '../data/20250920', '../data/20250918']
../data/20250908/ESPRE.2025-09-09T00:53:40.011/TOO_MOV_3I-ATLAS-v1_S1D_FINAL_A_2025-09-09T00:53:40.011.fits
../data/20250908/ESPRE.2025-09-09T00:32:28.054/TOO_MOV_3I-ATLAS-v1_S1D_FINAL_A_2025-09-09T00:32:28.054.fits
../data/20250908/ESPRE.2025-09-09T00:43:04.054/TOO_MOV_3I-ATLAS-v1_S1D_FINAL_A_2025-09-09T00:43:04.054.fits
[2, 2, 2]
../data/20250909/ESPRE.2025-09-09T23:45:51.143/TOO_MOV_3I-ATLAS-v1_S1D_FINAL_A_2025-09-09T23:45:51.143.fits
../data/20250909/ESPRE.2025-09-10T01:07:30.661/TOO_MOV_3I-ATLAS-v2_S1D_FINAL_A_2025-09-10T01:07:30.660.fits
../data/2